# Necesito asegurarme de que los círculos que utilizo en el matching no tengan cámaras de velocidad

In [1]:
import os
os.chdir("/Users/mariano/Documents/itam/tesis/speed-cameras/scripts/")

In [7]:
from ps_features_builder import PSFeaturesBuilder
from effect_estimation import EffectEstimation
from ps_matching import PSMatching
import geopandas as gpd
import pandas as pd
import pickle

In [3]:
PATH_DATA = "../data/"
PATHS = {
    'vialidades': os.path.join(PATH_DATA, 'vialidades.json'),
    'speed_cameras': os.path.join(PATH_DATA, 'fotocivicas-ubicacion-puntos', 'fotocivicas-ubicacion-puntos.shp'),
    'metro_coordinates': os.path.join(PATH_DATA, 'metro-station-coordinates.parquet'),
    'afluencia_metro': os.path.join(PATH_DATA, 'afluencia-metro-semanal.parquet'),
    'classified_incidents': os.path.join(PATH_DATA, 'classified-incidents.parquet'),
    'volumen_mensual': os.path.join(PATH_DATA, 'volumen-total-mensual.parquet'),
    'ps_feature_builders' : os.path.join(PATH_DATA, "pickle_objects", "ps_features_builder")
}

# grid_type, radio, n_circles
radios = [
    ("circular", 37.5, 25), # 0
    ("circular", 50, 15), # 1
    ("circular", 100, 20), # 2
    ("circular", 150, 15), # 3
    ("circular", 200, 20), # 4
    ("circular", 250, 25), # 5
]

object_name_format = "{grid_type}_{radio}_{n_circles}.pkl"

In [32]:
grid_type, radio, n_circles = radios[5]

ps_builder = PSFeaturesBuilder(
    paths=PATHS,
    grid_type=grid_type,
    circle_radius=radio,
    n_circles=n_circles
)
ps_builder.build()

# 2. Matching
psm = PSMatching(
    ps_features=ps_builder.ps_features,
    grid=ps_builder.grid,
    outcome=ps_builder.outcome,
    grid_type='circular',
    grid_size=radio
)
psm.build()

In [33]:
effect_estimator = EffectEstimation(
    outcome=psm.outcome,
    matched_grids=psm.matched_grids,
    inicio_operaciones=ps_builder.inicio_operaciones
)
effect_estimator.estimate_all()

In [34]:
effect_estimator.get_all_treatment_effects()

,outcome_type,variable,coeficiente,error_estandar,valor_p,valor_p_fe
0,total,total,-1.165362e-01,1.419570e-01,0.411689,0.527424
1,total,min,-1.690977e-01,9.302707e-02,0.069106,0.256300
2,total,pic,6.045331e-02,7.579000e-02,0.425078,0.456678
3,total,fcs,-7.891820e-03,5.722691e-03,0.167882,0.250092
4,tasas,total,-5.071540e-07,9.191143e-06,0.955996,0.963319
5,tasas,min,-6.850609e-06,5.976796e-06,0.251712,0.423724
6,tasas,pic,6.773197e-06,5.040621e-06,0.179038,0.264000
7,tasas,fcs,-4.297423e-07,3.825129e-07,0.261237,0.350891


In [35]:
# guardamos el builder
with open(os.path.join(PATH_DATA, "pickle_objects", "ps_features_builder", f"circular_{radio}_{n_circles}.pkl"), "wb") as f:
    pickle.dump(ps_builder, f)

In [49]:
control_ids = psm.matched_grids[psm.matched_grids.has_camera == 0].grid_id.tolist()
controles = psm.grid[psm.grid.grid_id.isin(control_ids)].drop(columns=["centroid"])
cameras = ps_builder.speed_cameras.rename(columns={"no":"camera_id"})[["camera_id", "geometry"]]

# Listo, ahora sí puedo estar tranquilo de que no hay efecto de los radares sobre nada
# me andaba cagando, pero ahora sí ya está todo listo.
gpd.sjoin(cameras, controles, predicate='within')

,camera_id,geometry,index_right,grid_id
